# Module 1B: Understanding DMF Costs

## Learning Objectives
- Query your DMF credit consumption from Account Usage views
- Compare scheduling strategies and their cost implications
- Apply cost optimization techniques to reduce DMF spend
- Build a cost monitoring query for ongoing visibility

## Key Concept: DMFs Use Serverless Compute

Data Metric Functions run on **serverless compute** -- Snowflake manages the infrastructure. You pay per execution in credits, not per warehouse hour.

| Factor | Impact on Cost |
|--------|---------------|
| Number of DMFs attached | More DMFs = more executions = more credits |
| Schedule type | TRIGGER_ON_CHANGES fires on every DML; cron fires on fixed intervals |
| Table size | Larger tables = longer DMF execution = more credits per run |
| DMF complexity | Simple COUNT(*) is cheap; complex JOINs or CROSS JOINs cost more |
| Number of columns | Each column-level DMF is a separate execution |

> **Rule of Thumb:** A simple system DMF (NULL_COUNT, ROW_COUNT) on a 1M-row table costs ~0.001-0.01 credits per execution. A complex custom DMF with JOINs can cost 10-100x more.

---

> **Role:** `CORP_DQ_ADMIN` | **Time:** ~20 minutes

> **What this does:** Sets your session context to the lab role, database, and warehouse.

In [ ]:
USE ROLE CORP_DQ_ADMIN;
USE DATABASE CORP_DWH;
USE WAREHOUSE COMPUTE_WH;

---
## 1B-a. How Much Are Your DMFs Costing?

> **Business Value:** Without cost visibility, teams add DMFs freely until the bill arrives. Proactive cost monitoring prevents surprise charges and enables informed decisions about which checks are worth paying for.

Snowflake tracks all DMF executions in the Account Usage view `DATA_QUALITY_MONITORING_USAGE_HISTORY`. This view has a 1-2 hour latency (like all Account Usage views).

> **What this does:** Queries total DMF credit consumption over the last 7 days, grouped by day. Shows your daily DQ monitoring cost trend.

In [ ]:
-- Total DMF credits consumed in last 7 days (grouped by day)
SELECT
    DATE_TRUNC('DAY', START_TIME) AS DAY,
    COUNT(*) AS EXECUTIONS,
    SUM(CREDITS_USED) AS TOTAL_CREDITS,
    ROUND(SUM(CREDITS_USED) * 3.0, 2) AS ESTIMATED_COST_USD
FROM SNOWFLAKE.ACCOUNT_USAGE.DATA_QUALITY_MONITORING_USAGE_HISTORY
WHERE START_TIME > DATEADD(DAY, -7, CURRENT_TIMESTAMP())
GROUP BY DAY
ORDER BY DAY;

> **What this does:** Breaks down DMF costs by table, so you can identify which tables are the most expensive to monitor.

In [ ]:
-- Credits consumed per table (which tables cost the most to monitor?)
SELECT
    TABLE_NAME,
    COUNT(*) AS TOTAL_EXECUTIONS,
    SUM(CREDITS_USED) AS TOTAL_CREDITS,
    ROUND(AVG(CREDITS_USED), 6) AS AVG_CREDIT_PER_EXECUTION,
    MAX(START_TIME) AS LAST_EXECUTION
FROM SNOWFLAKE.ACCOUNT_USAGE.DATA_QUALITY_MONITORING_USAGE_HISTORY
WHERE START_TIME > DATEADD(DAY, -7, CURRENT_TIMESTAMP())
GROUP BY TABLE_NAME
ORDER BY TOTAL_CREDITS DESC;

> **What this does:** Shows cost per individual DMF -- helps identify expensive custom DMFs that might need optimization.

In [ ]:
-- Credits per DMF (which specific checks are expensive?)
SELECT
    METRIC_NAME,
    TABLE_NAME,
    COUNT(*) AS EXECUTIONS,
    SUM(CREDITS_USED) AS TOTAL_CREDITS,
    ROUND(AVG(CREDITS_USED), 6) AS AVG_CREDIT_PER_RUN
FROM SNOWFLAKE.ACCOUNT_USAGE.DATA_QUALITY_MONITORING_USAGE_HISTORY
WHERE START_TIME > DATEADD(DAY, -7, CURRENT_TIMESTAMP())
GROUP BY METRIC_NAME, TABLE_NAME
ORDER BY TOTAL_CREDITS DESC
LIMIT 20;

---
## 1B-b. Scheduling Strategies and Cost Impact

> **Business Value:** Choosing the right schedule is the single biggest cost lever. TRIGGER_ON_CHANGES on a table updated every 5 minutes means 288 DMF runs/day. A cron schedule of once/hour means 24 runs/day -- 12x cheaper for the same quality insight.

| Strategy | When DMFs Run | Best For | Cost |
|----------|--------------|----------|------|
| `TRIGGER_ON_CHANGES` | Every DML (INSERT/UPDATE/DELETE) | Low-volume tables, critical data | HIGH if table changes often |
| `USING CRON '0 * * * *'` | Every hour (or custom cron) | High-volume tables, non-critical | PREDICTABLE |
| `USING CRON '0 8 * * *'` | Once daily at 8am | Batch loads, historical tables | LOW |
| Manual (no schedule) | Only when you explicitly trigger | Ad-hoc investigation | ZERO ongoing |

### Cost Formula
```
Monthly Cost = (DMF executions per day) x 30 x (avg credits per execution) x ($3/credit)
```

Example: 5 DMFs on a table with TRIGGER_ON_CHANGES, updated every 10 minutes:
- 5 DMFs x 144 triggers/day x 30 days x 0.005 credits = **108 credits/month = ~$324/month**

Same 5 DMFs with hourly cron:
- 5 DMFs x 24 runs/day x 30 days x 0.005 credits = **18 credits/month = ~$54/month**

> **What this does:** Shows the current schedule configuration for all your lab tables so you can see which strategy each uses.

In [ ]:
-- View current schedules on lab tables
SELECT
    REF_ENTITY_NAME AS TABLE_NAME,
    METRIC_NAME,
    SCHEDULE,
    SCHEDULE_STATUS
FROM TABLE(INFORMATION_SCHEMA.DATA_METRIC_FUNCTION_REFERENCES(
    REF_ENTITY_NAME => 'CORP_DWH.RAW.STG_CUSTOMERS_ERP',
    REF_ENTITY_DOMAIN => 'TABLE'
))
UNION ALL
SELECT REF_ENTITY_NAME, METRIC_NAME, SCHEDULE, SCHEDULE_STATUS
FROM TABLE(INFORMATION_SCHEMA.DATA_METRIC_FUNCTION_REFERENCES(
    REF_ENTITY_NAME => 'CORP_DWH.RAW.STG_TRANSACTIONS',
    REF_ENTITY_DOMAIN => 'TABLE'
))
ORDER BY TABLE_NAME, METRIC_NAME;

> **What this does:** Demonstrates the syntax for switching schedules. The commands are commented out to avoid changing your lab setup.

In [ ]:
-- Example: Switch a high-volume table from trigger to hourly cron
-- (Commented out - don't run unless you want to change lab behavior)
-- ALTER TABLE CORP_DWH.RAW.STG_TRANSACTIONS
--     SET DATA_METRIC_SCHEDULE = 'USING CRON 0 * * * * UTC';

-- To switch back:
-- ALTER TABLE CORP_DWH.RAW.STG_TRANSACTIONS
--     SET DATA_METRIC_SCHEDULE = 'TRIGGER_ON_CHANGES';

-- Summary of schedule options:
SELECT
    'TRIGGER_ON_CHANGES' AS STRATEGY, 'Runs on every DML' AS BEHAVIOR, 'Critical, low-volume' AS BEST_FOR
UNION ALL SELECT 'USING CRON 0 * * * * UTC', 'Runs every hour', 'High-volume, non-critical'
UNION ALL SELECT 'USING CRON 0 8 * * * UTC', 'Runs daily at 8am', 'Batch-loaded tables'
UNION ALL SELECT '5 MINUTES', 'Runs every 5 minutes', 'Near-real-time critical';

---
## 1B-c. Cost Optimization Tips

> **Business Value:** Smart optimization can reduce DMF costs by 50-80% without sacrificing coverage. The key insight: not every column needs real-time monitoring.

### The Tiered Monitoring Pattern

```
TIER 1 (Critical): TRIGGER_ON_CHANGES
  - Gold fact tables feeding live dashboards
  - Tables with regulatory SLAs (bank feeds, compliance data)
  
TIER 2 (Important): Hourly cron
  - Silver transformation tables
  - Customer-facing data
  
TIER 3 (Informational): Daily cron
  - Historical/archive tables
  - Reference/lookup tables
```

This maps directly to the `SLA_TIER` tags we create in Module 6!

### Quick Wins

| Strategy | Savings | Trade-off |
|----------|---------|----------|
| Use cron instead of TRIGGER_ON_CHANGES for non-critical tables | 50-90% | Delayed detection |
| Remove DMFs from static tables that never change | 100% | Must re-add if table becomes active |
| Use daily schedule for archive/historical tables | 90%+ | Next-day detection only |
| Monitor only key columns (not every STRING column) | 40-60% | May miss edge-case issues |

> **What this does:** Creates a cost projection that estimates your monthly DMF spend based on the last 24 hours of activity.

In [ ]:
-- Cost projection: estimate monthly spend based on last 24 hours
SELECT
    TABLE_NAME,
    COUNT(*) AS EXECUTIONS_LAST_24H,
    SUM(CREDITS_USED) AS CREDITS_LAST_24H,
    ROUND(SUM(CREDITS_USED) * 30, 4) AS PROJECTED_MONTHLY_CREDITS,
    ROUND(SUM(CREDITS_USED) * 30 * 3.0, 2) AS PROJECTED_MONTHLY_COST_USD
FROM SNOWFLAKE.ACCOUNT_USAGE.DATA_QUALITY_MONITORING_USAGE_HISTORY
WHERE START_TIME > DATEADD(HOUR, -24, CURRENT_TIMESTAMP())
GROUP BY TABLE_NAME
ORDER BY PROJECTED_MONTHLY_CREDITS DESC;

---
## 1B-d. Decision Framework: Is This DMF Worth the Cost?

Before attaching a new DMF, ask:

1. **What's the blast radius if this check fails?** (1 report vs entire pipeline)
2. **How quickly must we detect the issue?** (Minutes vs hours vs next day)
3. **How often does the source table change?** (Every 5 min vs daily batch)
4. **Is there a cheaper alternative?** (dbt test at build time vs always-on DMF)

### Cost vs Value Matrix

| Check | Annual Cost* | Annual Risk if Missing | Worth It? |
|-------|-------------|----------------------|----------|
| NULL_COUNT on NATIONAL_ID | ~$20 | $50K (compliance fine) | YES |
| FRESHNESS on bank feed | ~$30 | $100K (late reconciliation) | YES |
| BLANK_COUNT on optional NOTES column | ~$20 | $0 (cosmetic only) | NO |
| CHECK_DUPLICATES on Gold dim | ~$50 | $25K (double-billing) | YES |
| ROW_COUNT on archive table (never changes) | ~$10 | $0 (static data) | NO |

*Based on hourly cron, 0.005 credits/execution, $3/credit

---
## Checkpoint: Cost Visibility

> **What this does:** Verifies you can query DMF cost data. If results are empty, DMFs haven't run long enough for Account Usage to populate (1-2 hour latency).

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

print("=" * 60)
print("CHECKPOINT: DMF Cost Visibility")
print("=" * 60)

try:
    usage = session.sql("""
    SELECT COUNT(*) AS EXECUTIONS, COALESCE(SUM(CREDITS_USED), 0) AS TOTAL_CREDITS
    FROM SNOWFLAKE.ACCOUNT_USAGE.DATA_QUALITY_MONITORING_USAGE_HISTORY
    WHERE START_TIME > DATEADD(DAY, -7, CURRENT_TIMESTAMP())
    """).collect()
    
    executions = usage[0]['EXECUTIONS']
    credits = float(usage[0]['TOTAL_CREDITS'])
    
    if executions > 0:
        print(f"  [PASS] DMF usage data available")
        print(f"         Executions (7 days): {executions}")
        print(f"         Credits used: {credits:.6f}")
        print(f"         Estimated monthly: ${credits / 7 * 30 * 3:.2f}")
    else:
        print(f"  [INFO] No DMF usage data yet (Account Usage has 1-2 hour latency)")
        print(f"         This is expected if you just started the lab.")
        print(f"         Re-run this cell after completing Modules 2-4.")
except Exception as e:
    print(f"  [WARN] Cannot query Account Usage: {str(e)[:80]}")
    print(f"         You may need ACCOUNTADMIN or MONITOR USAGE privilege.")

print("=" * 60)

---
## Quiz: Test Your Knowledge

**Q1:** You have 10 DMFs on a table that receives Snowpipe data every 2 minutes. Using TRIGGER_ON_CHANGES, how many DMF executions per day?

**Q2:** Your monthly DMF bill is $500. The data team says "just monitor everything." What questions should you ask before adding 50 more DMFs?

**Q3:** A table is loaded once daily at 6am via a batch job. What schedule should its DMFs use and why?

**Q4:** The CHECK_AMOUNT_ANOMALIES DMF (Module 5) uses a CROSS JOIN and costs 10x more than NULL_COUNT. Is it worth keeping?

> **What this does:** Reveals quiz answers. Try answering first!

In [ ]:
print("""
QUIZ ANSWERS
============

Q1: 10 DMFs x 720 triggers/day (60min/2min x 24h) = 7,200 executions/day.
    At 0.005 credits each = 36 credits/day = $3,240/month.
    Switch to hourly cron = 10 x 24 = 240 executions/day = $108/month. Savings: 97%!

Q2: Questions to ask:
    - Which tables actually change frequently? (Skip static ones)
    - What's the business impact if issues go undetected for 1 hour? 1 day?
    - Are there existing dbt tests that already cover some checks?
    - Can we start with TIER 3 (daily) and promote only if issues are found?
    - What's the projected monthly cost?

Q3: Daily batch at 6am -> Use cron: 'USING CRON 0 7 * * * UTC' (1 hour after load).
    Why not TRIGGER_ON_CHANGES? It fires DURING the batch (possibly hundreds of
    INSERTs), causing hundreds of redundant DMF runs. One post-load check is enough.

Q4: Depends on blast radius. If it catches fraud ($10K+ per incident): YES.
    If it only catches data entry errors ($50 per fix): probably not at 10x cost.
    Optimization: Run it DAILY instead of hourly (saves 96%), or pre-compute
    mean/stddev in a task and reference cached values.
""")

---
## Summary

| Section | What You Learned |
|---------|-----------------|
| 1B-a | Query DMF credit consumption from Account Usage |
| 1B-b | TRIGGER_ON_CHANGES vs cron cost comparison (12x difference!) |
| 1B-c | Tiered monitoring pattern (map SLA_TIER tag to schedule) |
| 1B-d | Cost vs value decision framework |

**Key Takeaway:** The cheapest DMF is the one you don't need. Start with daily schedules, promote to hourly/trigger only for tables where detection speed justifies the cost.

---

**Next:** Open `2_SILVER_LAYER_DQ` to build custom DMFs for business-specific validation.